# 🧭 Compass -- AI-Powered Research & Analytics Agent

**Author:** Benedict Daxell Santoso  
**Course:** ISOM 260/900 -- AI for Business (Suffolk University)  
**Date:** March 2026

---

### What is Compass?

Compass is an autonomous AI research agent that answers business and analytics questions by **fetching real data** from live sources -- not by guessing.

Give it a company name, a topic, or a research question, and it will:

- 🔍 **Search Wikipedia** for background information
- 🌐 **Fetch any public webpage** and extract its content
- 🧮 **Calculate** metrics, growth rates, and comparisons
- 📰 **Pull latest news headlines** on any topic

It chains these tools together in **multi-step reasoning**, remembers your full conversation, and **never crashes** -- it explains errors gracefully instead.

### Architecture (The Agent Equation)

```
LLM (Claude) + Tools (4 Real APIs) + Reasoning Loop + Memory = Compass Agent
```

---

In [1]:
# ============================================================
# 📦 SETUP -- Install Dependencies
# ============================================================
!pip install -q anthropic requests beautifulsoup4 feedparser

In [ ]:
# ============================================================
# 🔑 CONFIGURATION -- Imports & Claude API Key
# ============================================================

import anthropic
import json
import re
import requests
from bs4 import BeautifulSoup
import feedparser
import time
from datetime import datetime

# ⚠️ PASTE YOUR ANTHROPIC API KEY BELOW
# Get one at: https://console.anthropic.com/settings/keys
# New accounts get $5 free credits -- more than enough for this project.
ANTHROPIC_API_KEY = "YOUR_API_KEY_HERE"

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
MODEL = "claude-sonnet-4-20250514"

print(f"✅ Claude configured successfully (model: {MODEL})")

✅ Claude configured successfully (model: claude-sonnet-4-20250514)


---

## 🛠️ Tool Definitions

Each tool is a **real API connection** with full error handling.

Every tool follows the same safety pattern:
- `try/except` wrapping to prevent crashes
- Graceful error messages for 404s, timeouts, and edge cases
- Text truncation to **3,000 characters** to prevent token overflow

---

In [3]:
# ============================================================
# 🔍 TOOL 1: Wikipedia Search
# ============================================================
# Connects to the real Wikipedia REST API to look up any topic.
# Returns a structured summary -- no hardcoded data.

def wikipedia_search(query):
    """Search Wikipedia for a topic and return its summary."""
    try:
        formatted_query = query.strip().replace(" ", "_")
        url = f"https://en.wikipedia.org/api/rest_v1/page/summary/{formatted_query}"

        response = requests.get(url, timeout=10, headers={"User-Agent": "Compass-Agent/1.0"})

        if response.status_code == 404:
            return f"No Wikipedia page found for '{query}'. Try a different search term."

        response.raise_for_status()
        data = response.json()

        title = data.get("title", "Unknown")
        extract = data.get("extract", "No summary available.")
        page_url = data.get("content_urls", {}).get("desktop", {}).get("page", "")

        result = f"Wikipedia: {title}\n\n{extract}"
        if page_url:
            result += f"\n\nSource: {page_url}"

        return result[:3000]

    except requests.exceptions.Timeout:
        return "Wikipedia request timed out. Please try again."
    except requests.exceptions.RequestException as e:
        return f"Error connecting to Wikipedia: {str(e)}"
    except Exception as e:
        return f"Unexpected error in Wikipedia search: {str(e)}"


# Quick self-test
print(wikipedia_search("Suffolk University"))

Wikipedia: Suffolk University

Suffolk University is a private university, Carnegie designated as "Professions-focused Undergraduate/Graduate-Doctorate Medium", in Boston, Massachusetts, United States. With 7,560 students on all campuses, it is the tenth-largest university in metropolitan Boston. It was founded as a law school in 1906 and named after its location in Suffolk County, Massachusetts. The university is also host to its namesake public opinion poll, the Suffolk University Political Research Center.

Source: https://en.wikipedia.org/wiki/Suffolk_University


In [4]:
# ============================================================
# 🌐 TOOL 2: Web Page Fetcher
# ============================================================
# Fetches and extracts the main text content from any public URL.
# Uses BeautifulSoup for HTML parsing. Handles errors gracefully.

def web_fetcher(url):
    """Fetch and extract main text content from a public web page."""
    try:
        if not url.startswith(("http://", "https://")):
            url = "https://" + url

        headers = {
            "User-Agent": "Mozilla/5.0 (Compass-Agent/1.0)",
            "Accept": "text/html"
        }
        response = requests.get(url, timeout=15, headers=headers)

        if response.status_code == 404:
            return f"Page not found (404): {url}"

        response.raise_for_status()

        soup = BeautifulSoup(response.text, "html.parser")

        # Remove non-content elements
        for tag in soup(["script", "style", "nav", "footer", "header"]):
            tag.decompose()

        text = soup.get_text(separator="\n", strip=True)

        lines = [line.strip() for line in text.splitlines() if line.strip()]
        clean_text = "\n".join(lines)

        if not clean_text:
            return f"Page loaded but no readable text content found at {url}"

        return f"Content from {url}:\n\n{clean_text[:3000]}"

    except requests.exceptions.Timeout:
        return f"Request timed out for {url}. The site may be slow or blocking requests."
    except requests.exceptions.ConnectionError:
        return f"Could not connect to {url}. Check the URL and try again."
    except requests.exceptions.RequestException as e:
        return f"Error fetching {url}: {str(e)}"
    except Exception as e:
        return f"Unexpected error fetching web page: {str(e)}"


# Quick self-test
result = web_fetcher("https://en.wikipedia.org/wiki/Python_(programming_language)")
print(result[:500])

Content from https://en.wikipedia.org/wiki/Python_(programming_language):

Python (programming language) - Wikipedia
Jump to content
From Wikipedia, the free encyclopedia
General-purpose programming language
This article
may contain
unverified
or
indiscriminate
information
in
embedded lists
.
Please help
clean up the lists
by removing items or incorporating them into the text of the article.
(
October 2025
)
Python
Paradigm
Multi-paradigm
:
object-oriented
,
[
1
]
procedural
(
imperative
),
func


In [5]:
# ============================================================
# 🧮 TOOL 3: Calculator
# ============================================================
# Evaluates mathematical expressions safely using Python.
# Sandboxed -- only math operations are allowed.

import math as _math

SAFE_MATH_NAMESPACE = {
    "__builtins__": {},
    "abs": abs, "round": round, "min": min, "max": max,
    "pow": pow, "sum": sum, "int": int, "float": float,
    "math": _math, "sqrt": _math.sqrt, "log": _math.log,
    "pi": _math.pi, "e": _math.e, "ceil": _math.ceil, "floor": _math.floor,
}

def calculator(expression):
    """Evaluate a mathematical expression and return the result."""
    try:
        expr = expression.strip()

        # Safety check -- reject code injection attempts
        dangerous_keywords = ["import", "exec", "eval", "open", "os.", "sys.", "__"]
        for keyword in dangerous_keywords:
            if keyword in expr.lower():
                return f"Expression rejected for safety: contains '{keyword}'"

        result = eval(expr, SAFE_MATH_NAMESPACE)
        return f"Calculation: {expr} = {result}"

    except ZeroDivisionError:
        return "Math error: Division by zero."
    except SyntaxError:
        return f"Invalid math expression: '{expression}'. Use valid Python math syntax."
    except Exception as e:
        return f"Calculator error: {str(e)}"


# Quick self-tests
print(calculator("2026 - 1906"))
print(calculator("(125000 * 0.15) - 8500"))
print(calculator("1 / 0"))

Calculation: 2026 - 1906 = 120
Calculation: (125000 * 0.15) - 8500 = 10250.0
Math error: Division by zero.


In [6]:
# ============================================================
# 📰 TOOL 4: News Search
# ============================================================
# Searches Google News RSS for the latest headlines on any topic.
# Returns real, current news -- no API key required.

def news_search(query):
    """Search for the latest news headlines on a topic via Google News RSS."""
    try:
        formatted_query = requests.utils.quote(query)
        rss_url = f"https://news.google.com/rss/search?q={formatted_query}&hl=en-US&gl=US&ceid=US:en"

        feed = feedparser.parse(rss_url)

        if not feed.entries:
            return f"No recent news found for '{query}'. Try broader search terms."

        headlines = []
        for i, entry in enumerate(feed.entries[:5], 1):
            title = entry.get("title", "No title")
            published = entry.get("published", "Unknown date")
            source = entry.get("source", {}).get("title", "Unknown source")
            headlines.append(f"{i}. {title} ({source}, {published})")

        result = f"Top headlines for '{query}':\n\n" + "\n\n".join(headlines)
        return result[:3000]

    except Exception as e:
        return f"Error searching news: {str(e)}"


# Quick self-test
print(news_search("artificial intelligence business"))

Top headlines for 'artificial intelligence business':

1. These Americans are quitting before AI can take their jobs—and starting their own businesses: 'I just wanted to be in control' - CNBC (CNBC, Fri, 27 Mar 2026 15:14:51 GMT)

2. NSF initiative aims to make every American worker, business and community AI-ready - National Science Foundation (.gov) (National Science Foundation (.gov), Wed, 25 Mar 2026 16:00:05 GMT)

3. Shield AI, a Start-Up Making Military Drones, Raises $2 Billion - The New York Times (The New York Times, Thu, 26 Mar 2026 11:57:21 GMT)

4. Where is AI moving beyond experimentation? 6 leaders on what's actually scaling - The World Economic Forum (The World Economic Forum, Wed, 25 Mar 2026 16:04:40 GMT)

5. Inside Meta’s play to turn 250M SMBs into AI customers - New York Post (New York Post, Wed, 25 Mar 2026 21:25:00 GMT)


In [7]:
# ============================================================
# 📋 TOOL REGISTRY
# ============================================================
# Maps tool names to functions and descriptions.
# Claude reads these descriptions to decide which tool to use.

TOOL_FUNCTIONS = {
    "wikipedia_search": wikipedia_search,
    "web_fetcher": web_fetcher,
    "calculator": calculator,
    "news_search": news_search,
}

TOOL_DESCRIPTIONS = """
You have access to the following tools. Use them to answer the user's questions accurately.

1. **wikipedia_search**
   - Description: Search Wikipedia for information about a company, person, topic, or concept.
   - Parameter: {"query": "search term"}
   - Example: {"tool": "wikipedia_search", "params": {"query": "Tesla Motors"}}

2. **web_fetcher**
   - Description: Fetch and extract the main text content from any public web page URL.
   - Parameter: {"url": "full URL"}
   - Example: {"tool": "web_fetcher", "params": {"url": "https://example.com"}}

3. **calculator**
   - Description: Evaluate any math expression. Use for arithmetic, percentages, age calculations, growth rates, etc.
   - Parameter: {"expression": "math expression"}
   - Example: {"tool": "calculator", "params": {"expression": "2026 - 1975"}}

4. **news_search**
   - Description: Search for the latest news headlines about any topic, company, or person.
   - Parameter: {"query": "search term"}
   - Example: {"tool": "news_search", "params": {"query": "Apple earnings"}}
"""

print(f"✅ {len(TOOL_FUNCTIONS)} tools registered: {', '.join(TOOL_FUNCTIONS.keys())}")

✅ 4 tools registered: wikipedia_search, web_fetcher, calculator, news_search


---

## 🧠 Agent Engine -- Reasoning Loop + Memory

This is the **core of Compass**. The reasoning loop:

1. Sends the user's question (+ conversation history) to Claude
2. If Claude decides to use a tool -- executes it -- feeds result back
3. Repeats until Claude has enough info to give a final answer
4. Appends everything to **conversation memory** for multi-turn context

**Built-in rate limit handling:** The engine automatically waits and retries if the API returns a rate limit error.

---

In [8]:
# ============================================================
# 🧠 COMPASS ENGINE -- Reasoning Loop + Memory (Claude-Powered)
# ============================================================

# Conversation memory -- persists across all queries in this session
conversation_history = []

SYSTEM_PROMPT = f"""You are Compass, an AI-powered research and analytics agent.

Your job is to answer business, research, and analytics questions by using your tools to fetch REAL data.
Do NOT guess or make up information -- always use a tool when you need facts, data, or calculations.

{TOOL_DESCRIPTIONS}

INSTRUCTIONS:
- When you need to use a tool, respond with ONLY a valid JSON object like:
  {{"tool": "tool_name", "params": {{"param": "value"}}}}
- Do NOT include any text before or after the JSON when calling a tool.
- You may need to call multiple tools in sequence (one per turn) to answer complex questions.
- When you have enough information to give a complete answer, respond with plain text (no JSON).
- Always be specific, cite your sources, and show your calculations.
- If a tool returns an error, explain the issue to the user -- never crash or ignore it.
- Be concise but thorough.
"""


def call_claude_with_retry(messages, max_retries=3):
    """
    Call Claude API with automatic retry on rate limit errors.
    Uses exponential backoff: waits 10s, then 20s, then 40s.
    """
    for attempt in range(max_retries):
        try:
            response = client.messages.create(
                model=MODEL,
                max_tokens=1024,
                system=SYSTEM_PROMPT,
                messages=messages
            )
            return response.content[0].text.strip()
        except anthropic.RateLimitError:
            wait_time = 10 * (2 ** attempt)
            print(f"   ⏳ Rate limited. Waiting {wait_time}s before retry ({attempt + 1}/{max_retries})...")
            time.sleep(wait_time)
        except anthropic.APIError as e:
            return f"Claude API error: {str(e)}"
        except Exception as e:
            return f"Unexpected error: {str(e)}"

    return "Rate limit persists after retries. Please wait a moment and try again."


def extract_json(text):
    """Try to extract a JSON tool call from the model's response."""
    text = text.strip()

    # Try direct JSON parse first
    try:
        parsed = json.loads(text)
        if isinstance(parsed, dict) and "tool" in parsed:
            return parsed
    except json.JSONDecodeError:
        pass

    # Try to find JSON within the text
    json_match = re.search(r'\{[^{}]*"tool"[^{}]*\}', text)
    if json_match:
        try:
            parsed = json.loads(json_match.group())
            if "tool" in parsed:
                return parsed
        except json.JSONDecodeError:
            pass

    return None


def run_compass(user_input, verbose=True):
    """
    Run the Compass agent with a user query.
    Uses the reasoning loop to chain tool calls and maintains conversation memory.
    """
    global conversation_history

    if verbose:
        print(f"\n{'='*60}")
        print(f"  USER: {user_input}")
        print(f"{'='*60}")

    # Add user message to memory
    conversation_history.append({"role": "user", "content": user_input})

    # Build messages for Claude (includes full conversation history)
    messages = list(conversation_history)

    # === REASONING LOOP ===
    max_steps = 8
    step = 0

    while step < max_steps:
        step += 1

        # Call Claude with automatic retry
        response_text = call_claude_with_retry(messages)

        # Check if Claude wants to call a tool
        tool_call = extract_json(response_text)

        if tool_call and "tool" in tool_call:
            tool_name = tool_call["tool"]
            tool_params = tool_call.get("params", {})

            if verbose:
                print(f"\n  🔧 Step {step}: Calling '{tool_name}' with {tool_params}")

            # Execute the tool
            if tool_name in TOOL_FUNCTIONS:
                func = TOOL_FUNCTIONS[tool_name]
                param_values = list(tool_params.values())
                if param_values:
                    tool_result = func(param_values[0])
                else:
                    tool_result = "No parameters provided for this tool."
            else:
                tool_result = f"Unknown tool: '{tool_name}'. Available: {', '.join(TOOL_FUNCTIONS.keys())}"

            if verbose:
                preview = tool_result[:200] + "..." if len(tool_result) > 200 else tool_result
                print(f"  📥 Result: {preview}")

            # Feed the tool result back to Claude
            messages.append({"role": "assistant", "content": response_text})
            messages.append({"role": "user", "content": f"Tool result for {tool_name}: {tool_result}"})

        else:
            # Claude gave a final text answer
            if verbose:
                print(f"\n  🧭 COMPASS:\n{response_text}")

            conversation_history.append({"role": "assistant", "content": response_text})
            return response_text

    # Safety: hit max steps
    fallback = "Reached maximum reasoning steps. Please try a more specific question."
    if verbose:
        print(f"\n  🧭 COMPASS: {fallback}")
    conversation_history.append({"role": "assistant", "content": fallback})
    return fallback


def reset_memory():
    """Clear conversation history to start fresh."""
    global conversation_history
    conversation_history = []
    print("🔄 Conversation memory cleared.")


print("✅ Compass engine loaded (Claude-powered). Use run_compass('your question') to start.")

✅ Compass engine loaded (Claude-powered). Use run_compass('your question') to start.


---

## 🎯 Demo Conversations

The following demos showcase all **four grading requirements**:

| # | Demo | Requirement Tested |
|---|------|--------------------|
| 1 | Company research + age calculation | ✅ Tool Use + Multi-step Planning |
| 2 | Follow-up question using "that company" | ✅ Memory / Multi-turn |
| 3 | Latest news search | ✅ Tool use with different tools |
| 4 | Nonexistent page lookup | ✅ Error Handling / Zero-crash |
| 5 | Complex multi-tool chain | ✅ All of the above combined |

---

In [9]:
# ============================================================
# 🎯 DEMO 1: Multi-Step Research (Wikipedia + Calculator)
# ============================================================
# Tests: Tool Use (the "Key Moment") + Multi-step Planning
# Expected: Agent calls wikipedia_search, then calculator

reset_memory()
run_compass("Look up Tesla Motors on Wikipedia and calculate how many years it has been operating as of 2026.")

🔄 Conversation memory cleared.

  USER: Look up Tesla Motors on Wikipedia and calculate how many years it has been operating as of 2026.

  🔧 Step 1: Calling 'wikipedia_search' with {'query': 'Tesla Motors'}
  📥 Result: Wikipedia: Tesla, Inc.

Tesla, Inc. is an American multinational automotive and clean energy company. Headquartered in Austin, Texas, it designs, manufactures, and sells battery electric vehicles (BEV...

  🧭 COMPASS:
I need to find Tesla's founding date to calculate how long it has been operating. Let me search for more specific information about when Tesla was founded.

{"tool": "wikipedia_search", "params": {"query": "Tesla Motors founding date history"}}


'I need to find Tesla\'s founding date to calculate how long it has been operating. Let me search for more specific information about when Tesla was founded.\n\n{"tool": "wikipedia_search", "params": {"query": "Tesla Motors founding date history"}}'

In [10]:
# ============================================================
# 🎯 DEMO 2: Memory / Multi-Turn Context
# ============================================================
# Tests: Agent must resolve "that company" from the previous answer.
# This proves conversation memory is working.

run_compass("Who founded that company? And how old is the main founder in 2026?")


  USER: Who founded that company? And how old is the main founder in 2026?

  🧭 COMPASS:
I need to search for information about Tesla's founders to answer your question.

{"tool": "wikipedia_search", "params": {"query": "Tesla founders Elon Musk Martin Eberhard"}}


'I need to search for information about Tesla\'s founders to answer your question.\n\n{"tool": "wikipedia_search", "params": {"query": "Tesla founders Elon Musk Martin Eberhard"}}'

In [11]:
# ============================================================
# 🎯 DEMO 3: News Search
# ============================================================
# Tests: Tool use with the news_search tool.

run_compass("What are the latest news headlines about artificial intelligence in business?")


  USER: What are the latest news headlines about artificial intelligence in business?

  🔧 Step 1: Calling 'news_search' with {'query': 'artificial intelligence business'}
  📥 Result: Top headlines for 'artificial intelligence business':

1. These Americans are quitting before AI can take their jobs—and starting their own businesses: 'I just wanted to be in control' - CNBC (CNBC, F...

  🧭 COMPASS:
Here are the latest news headlines about artificial intelligence in business:

**Top AI Business Headlines:**

1. **"These Americans are quitting before AI can take their jobs—and starting their own businesses: 'I just wanted to be in control'"** - CNBC (March 27, 2026)

2. **"NSF initiative aims to make every American worker, business and community AI-ready"** - National Science Foundation (March 25, 2026)

3. **"Shield AI, a Start-Up Making Military Drones, Raises $2 Billion"** - The New York Times (March 26, 2026)

4. **"Where is AI moving beyond experimentation? 6 leaders on what's actu

'Here are the latest news headlines about artificial intelligence in business:\n\n**Top AI Business Headlines:**\n\n1. **"These Americans are quitting before AI can take their jobs—and starting their own businesses: \'I just wanted to be in control\'"** - CNBC (March 27, 2026)\n\n2. **"NSF initiative aims to make every American worker, business and community AI-ready"** - National Science Foundation (March 25, 2026)\n\n3. **"Shield AI, a Start-Up Making Military Drones, Raises $2 Billion"** - The New York Times (March 26, 2026)\n\n4. **"Where is AI moving beyond experimentation? 6 leaders on what\'s actually scaling"** - The World Economic Forum (March 25, 2026)\n\n5. **"Inside Meta\'s play to turn 250M SMBs into AI customers"** - New York Post (March 25, 2026)\n\nThe headlines show several key trends: workers proactively starting their own businesses to stay ahead of AI disruption, government initiatives to prepare the workforce for AI integration, significant venture funding for AI s

In [12]:
# ============================================================
# 🎯 DEMO 4: Error Handling / Zero-Crash Reliability
# ============================================================
# Tests: Agent handles a nonexistent Wikipedia page gracefully.
# Expected: No Python exceptions -- agent explains the issue.

run_compass("Look up XYZZY Corporation on Wikipedia and tell me about their revenue.")


  USER: Look up XYZZY Corporation on Wikipedia and tell me about their revenue.

  🔧 Step 1: Calling 'wikipedia_search' with {'query': 'XYZZY Corporation'}
  📥 Result: No Wikipedia page found for 'XYZZY Corporation'. Try a different search term.

  🧭 COMPASS:
I wasn't able to find any information about "XYZZY Corporation" on Wikipedia. The search returned no results, which suggests that either:

1. This company doesn't have a Wikipedia page
2. The company name might be spelled differently
3. It might be a smaller or private company that doesn't meet Wikipedia's notability requirements
4. The company might not exist or might be fictional

Could you please:
- Double-check the spelling of the company name
- Provide any additional details about the company (industry, location, etc.)
- Or let me know if you'd like me to search for a different company

Without finding the company, I cannot provide information about their revenue.


'I wasn\'t able to find any information about "XYZZY Corporation" on Wikipedia. The search returned no results, which suggests that either:\n\n1. This company doesn\'t have a Wikipedia page\n2. The company name might be spelled differently\n3. It might be a smaller or private company that doesn\'t meet Wikipedia\'s notability requirements\n4. The company might not exist or might be fictional\n\nCould you please:\n- Double-check the spelling of the company name\n- Provide any additional details about the company (industry, location, etc.)\n- Or let me know if you\'d like me to search for a different company\n\nWithout finding the company, I cannot provide information about their revenue.'

In [13]:
# ============================================================
# 🎯 DEMO 5: Complex Multi-Tool Chain
# ============================================================
# Tests: All capabilities -- research, calculation, news in one query.

reset_memory()
run_compass("Look up Suffolk University on Wikipedia. Calculate how many years old it will be in 2026. Then find the latest news about Suffolk University.")

🔄 Conversation memory cleared.

  USER: Look up Suffolk University on Wikipedia. Calculate how many years old it will be in 2026. Then find the latest news about Suffolk University.

  🔧 Step 1: Calling 'wikipedia_search' with {'query': 'Suffolk University'}
  📥 Result: Wikipedia: Suffolk University

Suffolk University is a private university, Carnegie designated as "Professions-focused Undergraduate/Graduate-Doctorate Medium", in Boston, Massachusetts, United States...

  🔧 Step 2: Calling 'calculator' with {'expression': '2026 - 1906'}
  📥 Result: Calculation: 2026 - 1906 = 120

  🔧 Step 3: Calling 'news_search' with {'query': 'Suffolk University'}
  📥 Result: Top headlines for 'Suffolk University':

1. Jumbos win both games of doubleheader against Suffolk University - The Tufts Daily (The Tufts Daily, Thu, 26 Mar 2026 06:11:22 GMT)

2. SGA vice president r...

  🧭 COMPASS:
Based on my research, here's what I found about Suffolk University:

**Age Calculation:**
Suffolk University wa

'Based on my research, here\'s what I found about Suffolk University:\n\n**Age Calculation:**\nSuffolk University was founded in 1906 as a law school in Boston, Massachusetts. By 2026, it will be **120 years old**.\n\n**About Suffolk University:**\n- Private university located in Boston, Massachusetts\n- Founded in 1906 as a law school, named after Suffolk County\n- Currently has 7,560 students across all campuses\n- It\'s the tenth-largest university in metropolitan Boston\n- Carnegie classified as "Professions-focused Undergraduate/Graduate-Doctorate Medium"\n- Home to the Suffolk University Political Research Center, known for public opinion polling\n\n**Latest News:**\nRecent headlines about Suffolk University include:\n1. **Dining contract changes** - Suffolk ended its contract with Chartwells, resulting in 76 layoffs (March 2026)\n2. **Student government changes** - The SGA vice president resigned amid online controversy (March 2026)\n3. **Sports** - Tufts defeated Suffolk in a b

---

## 💬 Interactive Mode

Run the cell below to have a **live conversation** with Compass.

- Type your question and press Enter
- Type `quit` to exit
- Type `reset` to clear memory

---

In [14]:
# ============================================================
# 💬 INTERACTIVE MODE -- Talk to Compass
# ============================================================

reset_memory()

print("🧭 Welcome to Compass -- your AI research agent.")
print("   Type your question and press Enter.")
print("   Type 'quit' to exit or 'reset' to clear memory.\n")

while True:
    user_input = input("You: ").strip()

    if not user_input:
        continue
    if user_input.lower() == "quit":
        print("\n👋 Goodbye! Thanks for using Compass.")
        break
    if user_input.lower() == "reset":
        reset_memory()
        continue

    run_compass(user_input)

🔄 Conversation memory cleared.
🧭 Welcome to Compass -- your AI research agent.
   Type your question and press Enter.
   Type 'quit' to exit or 'reset' to clear memory.

You: Oil price right now

  USER: Oil price right now

  🔧 Step 1: Calling 'news_search' with {'query': 'oil price today current'}
  📥 Result: Top headlines for 'oil price today current':

1. Current price of oil as of March 27, 2026 - Fortune (Fortune, Fri, 27 Mar 2026 13:58:00 GMT)

2. The oil market is in 'backwardation' — Here’s what tha...

  🔧 Step 2: Calling 'web_fetcher' with {'url': 'https://fortune.com'}
  📥 Result: Content from https://fortune.com:

Fortune
Home
Latest
Fortune 500
Finance
Tech
Leadership
Lifestyle
Rankings
Multimedia
Economy
U.S. debt suddenly draws weaker demand as $10 trillion must be rolled o...

  🧭 COMPASS:
Let me search for more specific oil price information since the Fortune homepage didn't show the specific oil price article.

{"tool": "news_search", "params": {"query": "WTI crude o

---

## 📐 Architecture Summary

```
┌─────────────────────────────────────────────────┐
│                  USER QUERY                      │
│         "Look up Tesla and calculate..."         │
└──────────────────────┬──────────────────────────┘
                       │
                       ▼
┌─────────────────────────────────────────────────┐
│              🧠 CLAUDE (Brain)                   │
│         Reads query + tool descriptions          │
│         Decides: use a tool or answer?           │
└──────────────────────┬──────────────────────────┘
                       │
              ┌────────┴────────┐
              ▼                 ▼
     ┌──────────────┐  ┌──────────────┐
     │  🔧 TOOL CALL │  │ 💬 FINAL TEXT │
     │  (JSON)       │  │   (Answer)   │
     └───────┬──────┘  └──────────────┘
             │
             ▼
┌─────────────────────────────────────────────────┐
│           🛠️ TOOL EXECUTION                      │
│  wikipedia_search | web_fetcher | calculator     │
│  news_search                                     │
│  (Real APIs -- try/except -- truncate to 3000)   │
└──────────────────────┬──────────────────────────┘
                       │
                       ▼
┌─────────────────────────────────────────────────┐
│         🔁 REASONING LOOP                        │
│     Feed result back -- Claude decides next step │
│     Repeat until final answer (max 8 steps)      │
│     Auto-retry on rate limits                    │
└──────────────────────┬──────────────────────────┘
                       │
                       ▼
┌─────────────────────────────────────────────────┐
│         🧠 MEMORY                                │
│     conversation_history[] persists across turns  │
│     Enables multi-turn context                   │
└─────────────────────────────────────────────────┘
```

### Key Design Decisions

| Decision | Rationale |
|----------|-----------|
| Manual JSON tool calling | Matches Session 5 pattern; transparent for grading |
| 3,000-char truncation | Prevents token overflow on large web pages |
| try/except on every tool | Zero-crash reliability -- errors explain, never crash |
| Exponential backoff on rate limits | Agent survives API limits without user intervention |
| Conversation list as memory | Simple, effective -- matches class implementation |
| 8-step reasoning limit | Safety valve against infinite tool-calling loops |
| Sandboxed calculator | Cannot execute arbitrary code -- only math allowed |

---

### Built with
- **LLM:** Claude Sonnet 4 (Anthropic API)
- **Tools:** Wikipedia REST API, BeautifulSoup, Google News RSS, Python math
- **Environment:** Google Colab
- **Course:** ISOM 260/900 -- AI for Business, Suffolk University

*🧭 Compass -- Navigate any research question with AI.*